# VIX Replication Using ThetaData

Version: 0.2

Status:
- v0.2 methodology validated on five 2024 dates.
- Uses improved VIX-style expiration selection.
- Uses SPX AM settlement and SPXW PM settlement.
- Five-date validation max absolute error: about 0.165 VIX points.
- Five-date validation average absolute error: about 0.084 VIX points.

Remaining simplifications:
- Uses flat 5% risk-free rate.
- Uses approximate 9:30 AM / 4:00 PM settlement times.
- Does not yet use a historical Treasury/rate curve.
- Does not yet cache pulled option chains.

In [1]:
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from math import exp, sqrt

BASE_URL = "http://127.0.0.1:25510/v2"

CALC_TIME_MS = 57600000  # 16:00:00 ET
DEFAULT_RISK_FREE_RATE = 0.05

In [2]:
def td_get(endpoint, params, timeout=180):
    """
    Helper function to call ThetaData's local REST API.
    Theta Terminal must be running and logged in.
    """
    url = BASE_URL + endpoint

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()

    data = r.json()

    if data.get("header", {}).get("error_type") is not None:
        raise RuntimeError(data["header"])

    return data

In [3]:
def yyyymmdd_to_date(x):
    return datetime.strptime(str(int(x)), "%Y%m%d").date()

In [4]:
def list_expirations(root):
    data = td_get("/list/expirations", {"root": root})
    exps = data["response"]
    return sorted([int(x) for x in exps])


spx_exps = list_expirations("SPX")
spxw_exps = list_expirations("SPXW")

print("SPX expirations loaded:", len(spx_exps))
print("SPXW expirations loaded:", len(spxw_exps))

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=25510): Max retries exceeded with url: /v2/list/expirations?root=SPX (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001FA152BF380>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
# ============================================================
# Core VIX replication functions
# Paste this entire cell AFTER the list_expirations cell
# ============================================================

CALC_TIME_MS = 57600000          # 16:00:00 ET
DEFAULT_RISK_FREE_RATE = 0.05    # temporary simplification


def choose_vix_expirations(trade_date_yyyymmdd):
    trade_dt = yyyymmdd_to_date(trade_date_yyyymmdd)

    candidates = []

    # Standard SPX expirations
    for expi in spx_exps:
        exp_dt = yyyymmdd_to_date(expi)
        if exp_dt > trade_dt:
            candidates.append(("SPX", expi, (exp_dt - trade_dt).days))

    # SPXW Friday expirations
    for expi in spxw_exps:
        exp_dt = yyyymmdd_to_date(expi)
        if exp_dt > trade_dt and exp_dt.weekday() == 4:
            candidates.append(("SPXW", expi, (exp_dt - trade_dt).days))

    candidates = pd.DataFrame(candidates, columns=["root", "exp", "days"])
    candidates = candidates.sort_values("days")

    near = candidates[candidates["days"] <= 30].tail(1)
    next_ = candidates[candidates["days"] > 30].head(1)

    if near.empty or next_.empty:
        raise ValueError("Could not find near and next expirations around 30 days.")

    return pd.concat([near, next_]).reset_index(drop=True)


QUOTE_COLUMNS = [
    "ms_of_day", "bid_size", "bid_exchange", "bid", "bid_condition",
    "ask_size", "ask_exchange", "ask", "ask_condition", "date"
]


def get_chain_at_time(root, expi, trade_date, calc_time_ms):
    data = td_get(
        "/bulk_at_time/option/quote",
        {
            "root": root,
            "exp": expi,
            "start_date": trade_date,
            "end_date": trade_date,
            "ivl": calc_time_ms
        }
    )

    rows = []

    for item in data["response"]:
        contract = item["contract"]
        ticks = item["ticks"]

        if len(ticks) == 0:
            continue

        tick = ticks[-1]
        row = dict(zip(QUOTE_COLUMNS, tick))

        row["root"] = contract["root"]
        row["expiration"] = contract["expiration"]
        row["strike"] = contract["strike"] / 1000
        row["right"] = contract["right"]

        rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No data returned for {root} {expi} on {trade_date}")

    df["mid"] = (df["bid"] + df["ask"]) / 2

    return df


def minutes_to_expiry_approx(trade_date_yyyymmdd, exp_yyyymmdd, calc_time_ms):
    trade_dt = yyyymmdd_to_date(trade_date_yyyymmdd)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes = calc_time_ms / 1000 / 60
    expiry_minutes = 16 * 60  # 4:00 PM ET approximation

    calendar_days = (exp_dt - trade_dt).days

    return int(calendar_days * 1440 + expiry_minutes - calc_minutes)


def calc_single_term_variance(chain, minutes_to_expiry, r=0.05):
    T = minutes_to_expiry / 525600  # 365 * 1440 minutes

    df = chain.copy()

    # Basic quote cleaning
    df = df[(df["bid"] >= 0) & (df["ask"] > 0) & (df["ask"] >= df["bid"])]
    df = df[["strike", "right", "bid", "ask", "mid"]]

    calls = df[df["right"] == "C"].set_index("strike")
    puts = df[df["right"] == "P"].set_index("strike")

    common_strikes = calls.index.intersection(puts.index)

    cp = pd.DataFrame({
        "call_mid": calls.loc[common_strikes, "mid"],
        "put_mid": puts.loc[common_strikes, "mid"],
        "call_bid": calls.loc[common_strikes, "bid"],
        "put_bid": puts.loc[common_strikes, "bid"],
        "call_ask": calls.loc[common_strikes, "ask"],
        "put_ask": puts.loc[common_strikes, "ask"],
    }).sort_index()

    cp["abs_diff"] = (cp["call_mid"] - cp["put_mid"]).abs()
    atm_strike = cp["abs_diff"].idxmin()

    # Forward from put-call parity
    F = atm_strike + exp(r * T) * (
        cp.loc[atm_strike, "call_mid"] - cp.loc[atm_strike, "put_mid"]
    )

    # K0 = strike immediately below forward
    strikes = np.array(sorted(common_strikes))
    k0_candidates = strikes[strikes <= F]

    if len(k0_candidates) == 0:
        raise ValueError("No K0 found below forward.")

    K0 = k0_candidates[-1]

    selected_rows = []

    # OTM puts below K0
    put_strikes = sorted([k for k in puts.index if k < K0], reverse=True)
    zero_count = 0

    for k in put_strikes:
        bid = puts.loc[k, "bid"]
        ask = puts.loc[k, "ask"]

        if bid <= 0 or ask <= 0:
            zero_count += 1
            if zero_count >= 2:
                break
            continue

        zero_count = 0
        selected_rows.append({"strike": k, "Q": puts.loc[k, "mid"]})

    # OTM calls above K0
    call_strikes = sorted([k for k in calls.index if k > K0])
    zero_count = 0

    for k in call_strikes:
        bid = calls.loc[k, "bid"]
        ask = calls.loc[k, "ask"]

        if bid <= 0 or ask <= 0:
            zero_count += 1
            if zero_count >= 2:
                break
            continue

        zero_count = 0
        selected_rows.append({"strike": k, "Q": calls.loc[k, "mid"]})

    # At K0, use average of call and put midpoint
    k0_q = 0.5 * (calls.loc[K0, "mid"] + puts.loc[K0, "mid"])
    selected_rows.append({"strike": K0, "Q": k0_q})

    selected = (
        pd.DataFrame(selected_rows)
        .drop_duplicates("strike")
        .sort_values("strike")
        .reset_index(drop=True)
    )

    selected["delta_k"] = np.nan

    for i in range(len(selected)):
        if i == 0:
            selected.loc[i, "delta_k"] = (
                selected.loc[i + 1, "strike"] - selected.loc[i, "strike"]
            )
        elif i == len(selected) - 1:
            selected.loc[i, "delta_k"] = (
                selected.loc[i, "strike"] - selected.loc[i - 1, "strike"]
            )
        else:
            selected.loc[i, "delta_k"] = (
                selected.loc[i + 1, "strike"] - selected.loc[i - 1, "strike"]
            ) / 2

    selected["contribution"] = (
        selected["delta_k"]
        / (selected["strike"] ** 2)
        * exp(r * T)
        * selected["Q"]
    )

    variance = (
        (2 / T) * selected["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "vol": sqrt(max(variance, 0)) * 100,
        "T": T,
        "minutes": minutes_to_expiry,
        "F": F,
        "K0": K0,
        "selected_options": selected
    }


def interpolate_to_30d(term_df):
    near = term_df.iloc[0]
    next_ = term_df.iloc[1]

    M1 = near["minutes"]
    M2 = next_["minutes"]

    M30 = 30 * 1440
    M365 = 365 * 1440

    T1 = M1 / M365
    T2 = M2 / M365

    var1 = near["variance"]
    var2 = next_["variance"]

    variance_30d = (
        T1 * var1 * ((M2 - M30) / (M2 - M1))
        + T2 * var2 * ((M30 - M1) / (M2 - M1))
    ) * (M365 / M30)

    vix = 100 * sqrt(max(variance_30d, 0))

    return vix


def calculate_vix_for_date(trade_date, calc_time_ms=CALC_TIME_MS, r=DEFAULT_RISK_FREE_RATE):
    terms = choose_vix_expirations(trade_date)

    chains = []

    for _, row in terms.iterrows():
        print("Pulling:", row["root"], int(row["exp"]))

        chain = get_chain_at_time(
            root=row["root"],
            expi=int(row["exp"]),
            trade_date=trade_date,
            calc_time_ms=calc_time_ms
        )

        chain["days"] = row["days"]
        chains.append(chain)

    raw_options = pd.concat(chains, ignore_index=True)

    term_results = []

    for _, row in terms.iterrows():
        root = row["root"]
        expi = int(row["exp"])

        chain = raw_options[
            (raw_options["root"] == root) &
            (raw_options["expiration"] == expi)
        ]

        mins = minutes_to_expiry_approx(trade_date, expi, calc_time_ms)

        result = calc_single_term_variance(chain, mins, r=r)

        term_results.append({
            "root": root,
            "exp": expi,
            "days": row["days"],
            "minutes": mins,
            "variance": result["variance"],
            "vol": result["vol"],
            "F": result["F"],
            "K0": result["K0"],
            "num_selected_options": len(result["selected_options"])
        })

    term_df = pd.DataFrame(term_results)

    vix_value = interpolate_to_30d(term_df)

    return {
        "trade_date": trade_date,
        "calc_time_ms": calc_time_ms,
        "vix": vix_value,
        "term_df": term_df,
        "raw_options": raw_options
    }


def load_cboe_vix_history():
    cboe_url = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"
    cboe_vix = pd.read_csv(cboe_url)
    cboe_vix["DATE"] = pd.to_datetime(cboe_vix["DATE"])
    cboe_vix["date_int"] = cboe_vix["DATE"].dt.strftime("%Y%m%d").astype(int)
    return cboe_vix


def compare_to_cboe_close(trade_date, replicated_vix):
    cboe_vix = load_cboe_vix_history()

    row = cboe_vix[cboe_vix["date_int"] == int(trade_date)]

    if row.empty:
        raise ValueError(f"No Cboe VIX close found for {trade_date}")

    official_vix_close = float(row["CLOSE"].iloc[0])
    diff = replicated_vix - official_vix_close

    return {
        "date": trade_date,
        "replicated_vix": replicated_vix,
        "official_vix_close": official_vix_close,
        "difference": diff,
        "abs_difference": abs(diff)
    }

In [ ]:
result = calculate_vix_for_date(20240116)

print("Calculated VIX:", round(result["vix"], 4))
display(result["term_df"])

comparison = compare_to_cboe_close(20240116, result["vix"])
comparison

In [ ]:
# ============================================================
# v0.2 VIX calculation using improved expiration selection
# ============================================================

def choose_vix_expirations_v2(trade_date_yyyymmdd, calc_time_ms=CALC_TIME_MS):
    """
    Production version of the v0.2 expiration selector.
    
    Uses:
    - Friday expirations only
    - More than 23 and less than 37 days to expiry
    - SPX for standard monthly third-Friday expirations
    - SPXW for weekly expirations
    """
    candidates, selected = choose_vix_expirations_v2_diagnostic(
        trade_date_yyyymmdd,
        calc_time_ms=calc_time_ms
    )

    if len(selected) < 2:
        raise ValueError(f"Could not find two VIX expirations for {trade_date_yyyymmdd}")

    return selected.reset_index(drop=True)


def calculate_vix_for_date_v2(trade_date, calc_time_ms=CALC_TIME_MS, r=DEFAULT_RISK_FREE_RATE):
    """
    v0.2 VIX-style calculation.

    Improvements versus v0.1:
    - Uses VIX-style 23-to-37-day expiration selection.
    - Avoids duplicate SPX/SPXW expirations on the same calendar date.
    - Uses AM settlement timing for SPX.
    - Uses PM settlement timing for SPXW.
    """

    terms = choose_vix_expirations_v2(trade_date, calc_time_ms=calc_time_ms)

    chains = []

    for _, row in terms.iterrows():
        print("Pulling:", row["root"], int(row["exp"]))

        chain = get_chain_at_time(
            root=row["root"],
            expi=int(row["exp"]),
            trade_date=trade_date,
            calc_time_ms=calc_time_ms
        )

        chain["days"] = row["days"]
        chains.append(chain)

    raw_options = pd.concat(chains, ignore_index=True)

    term_results = []

    for _, row in terms.iterrows():
        root = row["root"]
        expi = int(row["exp"])
        mins = int(row["minutes"])

        chain = raw_options[
            (raw_options["root"] == root) &
            (raw_options["expiration"] == expi)
        ]

        result = calc_single_term_variance(chain, mins, r=r)

        term_results.append({
            "root": root,
            "exp": expi,
            "days": row["days"],
            "minutes": mins,
            "variance": result["variance"],
            "vol": result["vol"],
            "F": result["F"],
            "K0": result["K0"],
            "num_selected_options": len(result["selected_options"])
        })

    term_df = pd.DataFrame(term_results)

    vix_value = interpolate_to_30d(term_df)

    return {
        "trade_date": trade_date,
        "calc_time_ms": calc_time_ms,
        "vix": vix_value,
        "term_df": term_df,
        "raw_options": raw_options
    }

In [ ]:
result_v2 = calculate_vix_for_date_v2(20240116)

print("v0.2 Calculated VIX:", round(result_v2["vix"], 4))
display(result_v2["term_df"])

comparison_v2 = compare_to_cboe_close(20240116, result_v2["vix"])
comparison_v2

In [ ]:
test_dates = [
    20240116,
    20240215,
    20240315,
    20240415,
    20240515
]

v2_test_results = []

for d in test_dates:
    print("\n====================")
    print("Calculating:", d)

    try:
        result = calculate_vix_for_date_v2(d)
        comparison = compare_to_cboe_close(d, result["vix"])

        row = {
            "date": d,
            "replicated_vix_v2": result["vix"],
            "official_vix_close": comparison["official_vix_close"],
            "difference": comparison["difference"],
            "abs_difference": comparison["abs_difference"],
            "near_root": result["term_df"].iloc[0]["root"],
            "near_exp": result["term_df"].iloc[0]["exp"],
            "near_days": result["term_df"].iloc[0]["days"],
            "near_minutes": result["term_df"].iloc[0]["minutes"],
            "near_vol": result["term_df"].iloc[0]["vol"],
            "next_root": result["term_df"].iloc[1]["root"],
            "next_exp": result["term_df"].iloc[1]["exp"],
            "next_days": result["term_df"].iloc[1]["days"],
            "next_minutes": result["term_df"].iloc[1]["minutes"],
            "next_vol": result["term_df"].iloc[1]["vol"],
            "error": None
        }

    except Exception as e:
        print("FAILED:", d, e)

        row = {
            "date": d,
            "replicated_vix_v2": np.nan,
            "official_vix_close": np.nan,
            "difference": np.nan,
            "abs_difference": np.nan,
            "error": str(e)
        }

    v2_test_results.append(row)

v2_test_results_df = pd.DataFrame(v2_test_results)

display(v2_test_results_df)

In [ ]:
# ============================================================
# v0.3 diagnostic: test ThetaData v3 interest-rate REST endpoint
# This does NOT modify the VIX calculation yet.
# ============================================================

import requests
import pandas as pd

BASE_URL_V3 = "http://127.0.0.1:25503/v3"


def td_v3_get(endpoint, params, timeout=60):
    """
    Helper for ThetaData v3 REST API.
    Rates use v3 / port 25503, unlike our option code which uses v2 / port 25510.
    """
    url = BASE_URL_V3 + endpoint

    r = requests.get(url, params=params, timeout=timeout)
    print("URL:", r.url)
    print("Status code:", r.status_code)

    r.raise_for_status()

    data = r.json()
    return data


# Test with SOFR first because ThetaData's docs explicitly use symbol=SOFR
rate_test_data = td_v3_get(
    "/interest_rate/history/eod",
    {
        "symbol": "SOFR",
        "start_date": "20240116",
        "end_date": "20240119",
        "format": "json"
    }
)

print("Raw response keys:", rate_test_data.keys())
print("Raw response:")
print(rate_test_data)

In [19]:
# ============================================================
# Core VIX math functions
# Uses v3 data access through the current get_chain_at_time()
# ============================================================

import math
import numpy as np
import pandas as pd
from datetime import date, datetime

MINUTES_PER_DAY = 1440
MINUTES_PER_YEAR = 525600
MINUTES_30_DAYS = 43200


def yyyymmdd_to_date(x):
    """
    Convert 20240116 to datetime.date(2024, 1, 16).
    """
    s = str(int(x))
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))


def is_third_friday(dt):
    """
    True if date is the standard monthly SPX expiration Friday.
    """
    return dt.weekday() == 4 and 15 <= dt.day <= 21


def preferred_root_for_expiration(exp_yyyymmdd):
    """
    Prefer SPX for standard monthly third-Friday expirations.
    Otherwise use SPXW.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if is_third_friday(exp_dt) and exp_yyyymmdd in spx_exps:
        return "SPX"

    if exp_yyyymmdd in spxw_exps:
        return "SPXW"

    if exp_yyyymmdd in spx_exps:
        return "SPX"

    raise ValueError(f"Expiration {exp_yyyymmdd} not found in SPX or SPXW expiration lists")


def settlement_minutes_after_midnight_et(root):
    """
    Approximate VIX methodology settlement timing.

    SPX monthly options are AM-settled.
    SPXW weekly options are PM-settled.
    """
    if root == "SPX":
        return 9 * 60 + 30       # 9:30 AM ET

    if root == "SPXW":
        return 16 * 60           # 4:00 PM ET

    raise ValueError(f"Unknown root: {root}")


def minutes_to_expiry_vix_method(trade_date, exp_yyyymmdd, root, calc_time_ms):
    """
    Minutes from calculation time to option settlement time.
    """
    trade_dt = yyyymmdd_to_date(trade_date)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes_after_midnight = int(calc_time_ms // 60000)
    settlement_minutes = settlement_minutes_after_midnight_et(root)

    days_diff = (exp_dt - trade_dt).days

    return days_diff * MINUTES_PER_DAY + settlement_minutes - calc_minutes_after_midnight


def choose_vix_expirations_v2(trade_date, calc_time_ms=CALC_TIME_MS):
    """
    Select the two VIX-style expirations.

    Uses Friday expirations with roughly 23 to 37 days to expiry.
    Chooses the first two valid expirations.
    """
    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))

    candidates = []

    for exp in all_exps:
        exp_dt = yyyymmdd_to_date(exp)

        # VIX uses Friday expirations for the standard 30-day index
        if exp_dt.weekday() != 4:
            continue

        root = preferred_root_for_expiration(exp)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms
        )

        days = minutes / MINUTES_PER_DAY

        if days > 23 and days < 37:
            candidates.append({
                "root": root,
                "expiration": exp,
                "minutes": minutes,
                "days": days
            })

    candidates = sorted(candidates, key=lambda x: x["minutes"])

    if len(candidates) < 2:
        raise ValueError(f"Could not find two valid VIX expirations for {trade_date}")

    return candidates[0], candidates[1]


def _prepare_call_put_tables(chain):
    """
    Create call and put tables indexed by strike.
    """
    df = chain.copy()

    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["mid"] = pd.to_numeric(df["mid"], errors="coerce")

    df = df.dropna(subset=["strike", "bid", "ask", "mid", "right"])
    df = df[df["ask"] >= 0]
    df = df[df["bid"] >= 0]

    calls = (
        df[df["right"] == "C"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    puts = (
        df[df["right"] == "P"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    return calls, puts


def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Walk away from ATM.
    Keep options with positive bid.
    Stop after two consecutive zero-bid options.
    """
    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1
            if consecutive_zero_bids >= 2:
                break
            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style variance for one expiration.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)

    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    parity_rows = []

    for K in common_strikes:
        call_mid = calls.loc[K, "mid"]
        put_mid = puts.loc[K, "mid"]
        diff = abs(call_mid - put_mid)

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "abs_call_put_diff": diff
        })

    parity_df = pd.DataFrame(parity_rows)
    min_row = parity_df.loc[parity_df["abs_call_put_diff"].idxmin()]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    # OTM puts below K0, moving downward from K0
    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    # OTM calls above K0, moving upward from K0
    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    # K0 option uses average of call and put mid
    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm
        ],
        ignore_index=True
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K
    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options
    }


def interpolate_to_30d(term_df):
    """
    Interpolate near and next term variances to 30 calendar days.
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target = MINUTES_30_DAYS

    interpolated_variance = (
        T1 * var1 * ((N2 - target) / (N2 - N1))
        + T2 * var2 * ((target - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target)

    return 100 * math.sqrt(interpolated_variance)


def calculate_vix_for_date_v3(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    r=DEFAULT_RISK_FREE_RATE,
    return_raw_options=True
):
    """
    Calculate a replicated 30-day VIX value for one trade date.
    Uses v3 get_chain_at_time().
    """
    near_exp, next_exp = choose_vix_expirations_v2(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    near_chain = get_chain_at_time(
        root=near_exp["root"],
        expi=near_exp["expiration"],
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    next_chain = get_chain_at_time(
        root=next_exp["root"],
        expi=next_exp["expiration"],
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    near_calc = calc_single_term_variance(
        chain=near_chain,
        minutes_to_expiry=near_exp["minutes"],
        r=r
    )

    next_calc = calc_single_term_variance(
        chain=next_chain,
        minutes_to_expiry=next_exp["minutes"],
        r=r
    )

    term_df = pd.DataFrame([
        {
            "term": "near",
            "root": near_exp["root"],
            "expiration": near_exp["expiration"],
            "minutes": near_exp["minutes"],
            "days": near_exp["days"],
            "variance": near_calc["variance"],
            "F": near_calc["F"],
            "K0": near_calc["K0"],
            "num_options": near_calc["num_options"]
        },
        {
            "term": "next",
            "root": next_exp["root"],
            "expiration": next_exp["expiration"],
            "minutes": next_exp["minutes"],
            "days": next_exp["days"],
            "variance": next_calc["variance"],
            "F": next_calc["F"],
            "K0": next_calc["K0"],
            "num_options": next_calc["num_options"]
        }
    ])

    vix = interpolate_to_30d(term_df)

    result = {
        "trade_date": trade_date,
        "vix": vix,
        "term_df": term_df,
        "near_calc": near_calc,
        "next_calc": next_calc
    }

    if return_raw_options:
        result["near_chain"] = near_chain
        result["next_chain"] = next_chain

    return result


# Temporary alias so old test cells still work
calculate_vix_for_date_v2 = calculate_vix_for_date_v3

In [20]:
result_v3_20240116 = calculate_vix_for_date_v3(
    trade_date=20240116,
    calc_time_ms=CALC_TIME_MS,
    r=DEFAULT_RISK_FREE_RATE
)

print("Replicated VIX using v3 data:")
print(result_v3_20240116["vix"])

print("\nTerm details:")
display(result_v3_20240116["term_df"])

URL: http://127.0.0.1:25503/v3/option/history/quote?symbol=SPXW&expiration=2024-02-09&strike=%2A&right=both&start_date=2024-01-16&end_date=2024-01-16&start_time=16%3A00%3A00.000&end_time=16%3A00%3A00.000&interval=1m&format=json
Status code: 200
URL: http://127.0.0.1:25503/v3/option/history/quote?symbol=SPX&expiration=2024-02-16&strike=%2A&right=both&start_date=2024-01-16&end_date=2024-01-16&start_time=16%3A00%3A00.000&end_time=16%3A00%3A00.000&interval=1m&format=json
Status code: 200
Replicated VIX using v3 data:
13.831907052437831

Term details:


,term,root,expiration,minutes,days,variance,F,K0,num_options
0,near,SPXW,20240209,34560,24.000000,0.018789,4781.504940,4780.0,161
1,next,SPX,20240216,44250,30.729167,0.019165,4783.694516,4780.0,351
